# Public GRMHD dump validation against ipole/grtrans-style output

This notebook is the external validation path for `blackhole-sim` v0.5.0. It is designed for one selected public GRMHD HDF5 dump, then compares this project's Stokes image cube against an `ipole` image file produced from the same dump/camera/frequency/scaling.

The bundled manifest points at the Illinois Simulation Data Products v3 GRMHD collection. Select a concrete HDF5 dump from the portal, record its URL/SHA-256, and use the CLI cells below.


## 1. Install local package


In [ ]:
%pip install -e ..[dev,hdf5]


## 2. Write the public-data manifest

This records the source collection, target model family, citation, license, and adapter choice.


In [ ]:
!blackhole-public-dump --write-manifest --manifest ../data/public/illinois_v3_manifest.json
!cat ../data/public/illinois_v3_manifest.json


## 3. Download a selected dump

Replace `SELECTED_DUMP_URL` after choosing one concrete HDF5 dump from the public portal. Keep the SHA-256 in git or your evidence pack.


In [ ]:
SELECTED_DUMP_URL = "REPLACE_WITH_SELECTED_PUBLIC_HDF5_URL"
SELECTED_SHA256 = None  # fill after first download for reproducibility
if SELECTED_DUMP_URL.startswith("http"):
    sha_arg = f" --sha256 {SELECTED_SHA256}" if SELECTED_SHA256 else ""
    !blackhole-public-dump --download-url {SELECTED_DUMP_URL} --download-output ../data/public/selected_dump.h5 {sha_arg}
else:
    print("Set SELECTED_DUMP_URL first.")


## 4. Verify HDF5 schema and adapter mapping


In [ ]:
!blackhole-public-dump --verify ../data/public/selected_dump.h5 --adapter harm --report ../out/public_dump_verification.json
!cat ../out/public_dump_verification.json


## 5. Render our polarized Stokes image


In [ ]:
!blackhole-render-stokes \
  --snapshot ../data/public/selected_dump.h5 \
  --adapter harm \
  --width 64 --height 64 \
  --mass m87 \
  --rho-scale 1e-18 --b-scale 30 \
  --output ../out/ours_stokes.npz \
  --preview ../out/ours_stokes_preview.png


## 6. Generate an ipole parameter file

Build `ipole` separately, then run this parameter file against the same dump. Keep the exact ipole commit hash in your evidence pack.


In [ ]:
!blackhole-external-validation write-ipole-par \
  --dump ../data/public/selected_dump.h5 \
  --outfile ../out/ipole_image.h5 \
  --par ../out/ipole_validation.par \
  --nx 64 --ny 64 \
  --freq-hz 230e9 \
  --mbh-msun 6.2e9 \
  --m-unit-g 1e25 \
  --thetacam 17 \
  --fov-muas 160
!cat ../out/ipole_validation.par


## 7. Run ipole, then compare

Set `IPOLE_BIN` to your local `ipole` executable. The comparison fails intentionally if the image shapes or Stokes cubes diverge beyond the chosen tolerance.


In [ ]:
IPOLE_BIN = "/path/to/ipole"
if IPOLE_BIN.startswith("/path/to"):
    print("Set IPOLE_BIN first.")
else:
    !blackhole-external-validation run-ipole --ipole {IPOLE_BIN} --par ../out/ipole_validation.par
    !blackhole-external-validation compare --ours ../out/ours_stokes.npz --ipole ../out/ipole_image.h5 --report ../out/ipole_comparison.json --rtol-l1 0.2
    !cat ../out/ipole_comparison.json
